# Dialogue Summarizer — Evaluation on Kaggle GPU

Runs ROUGE evaluation on the DialogSum test split (819 examples).
Compares the fine-tuned LoRA adapter (`rotemso23/dialogsum-phi3-lora`) against the zero-shot baseline.

**Before running:**
1. Set Settings → Accelerator → **GPU T4 x2** (or P100)
2. Add your secrets in Settings → Secrets:
   - `HF_TOKEN` — HuggingFace token (read permission)
   - `GITHUB_TOKEN` — GitHub Personal Access Token (classic, `public_repo` scope)

**Expected runtime:** ~30–60 minutes (two inference passes over 819 examples).

## 1. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found! Enable GPU in Settings → Accelerator"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Install dependencies

In [ ]:
# Kaggle already has torch, so we skip it to avoid version conflicts
!pip install -q \
    "datasets>=2.0.0" \
    "transformers>=4.40.0" \
    "peft>=0.19.0" \
    "bitsandbytes>=0.43.0" \
    "accelerate>=0.30.0" \
    "rouge-score==0.1.2" \
    "python-dotenv==1.0.1"
print("Dependencies installed.")

## 3. Clone the repo

In [ ]:
import os

REPO_URL = "https://github.com/rotemso23/dialogue-summarizer.git"
REPO_DIR = "/kaggle/working/dialogue-summarizer"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

## 4. Set secrets

Reads `HF_TOKEN` and `GITHUB_TOKEN` from Kaggle Secrets (Settings → Secrets on the right sidebar).

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()

os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
github_token = secrets.get_secret("GITHUB_TOKEN")

# Write to .env so evaluate.py can find it via python-dotenv
with open(".env", "w") as f:
    f.write(f'HF_TOKEN={os.environ["HF_TOKEN"]}\n')

print("Secrets loaded.")

## 5. Run evaluation

This runs two full inference passes over the 819-example test split:
1. Fine-tuned model (`rotemso23/dialogsum-phi3-lora`)
2. Zero-shot baseline (same base model, no adapter)

Results are saved to `evaluation_results_<timestamp>.json`.

In [ ]:
!PYTHONPATH=/kaggle/working/dialogue-summarizer python src/evaluate.py

## 6. View results

In [ ]:
import json
import glob

result_files = sorted(glob.glob("evaluation_results_*.json"))
assert result_files, "No evaluation_results_*.json found — did evaluate.py run successfully?"
latest = result_files[-1]
print(f"Reading: {latest}\n")

with open(latest) as f:
    results = json.load(f)

print(f"{'Metric':<12} {'Baseline':>10} {'Fine-tuned':>12} {'Delta':>10}")
print("-" * 52)
for k in ["rouge1", "rouge2", "rougeL"]:
    base_val = results["baseline"][k]
    ft_val = results["fine_tuned"][k]
    delta = ft_val - base_val
    print(f"{k:<12} {base_val:>10.4f} {ft_val:>12.4f} {delta:>+10.4f}")

## 7. Commit results to GitHub

In [ ]:
!git config user.email "rotemso23@gmail.com"
!git config user.name "rotemso23"

!git add evaluation_results_*.json
!git commit -m "Add evaluation results (ROUGE scores on DialogSum test split)"
!git push https://{github_token}@github.com/rotemso23/dialogue-summarizer.git master

## 8. Shut down kernel

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)